#Disponibilidad planta

In [0]:
%pip install pydataxm

In [0]:
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
import os

import pandas as pd
from pydataxm.pydatasimem import ReadSIMEM

DATASET_ID = "9E77E5"
BRONZE_TABLE = "observatorio_dev.bronze.disponibilidad_plantas"
LANDING_PATH = "/Volumes/observatorio_dev/landing/raw_files/disponibilidad_plantas.json.gz"
HISTORICAL_START_DATE = date(2026, 1, 1)
LOOKBACK_DAYS = 45

# Umbral operativo, no una garantía contractual de SIMEM.
# La corrida del 16-jul mostró 15 días de rezago.
MAXIMUM_ACCEPTED_LAG_DAYS = 20

In [0]:
fecha_fin = datetime.now(ZoneInfo("America/Bogota")).date()

if not spark.catalog.tableExists(BRONZE_TABLE):
    bronze_max_date = None
else:
    bronze_max_date = spark.sql(
        f"SELECT MAX(CAST(fecha_hora AS DATE)) AS max_date FROM {BRONZE_TABLE}"
    ).first()["max_date"]

if bronze_max_date is None:
    fecha_inicio = HISTORICAL_START_DATE
    execution_mode = "BACKFILL"
else:
    fecha_inicio = max(
        HISTORICAL_START_DATE,
        bronze_max_date - timedelta(days=LOOKBACK_DAYS),
    )
    execution_mode = "INCREMENTAL"

fecha_inicio_str = fecha_inicio.strftime("%Y-%m-%d")
fecha_fin_str = fecha_fin.strftime("%Y-%m-%d")

print(f"Modo de ejecución: {execution_mode}")
print(f"Fecha máxima actual en Bronze: {bronze_max_date}")
print(f"Rango solicitado a SIMEM: {fecha_inicio_str} a {fecha_fin_str}")

In [0]:
df_disponibilidad_planta = ReadSIMEM(
    DATASET_ID,
    fecha_inicio_str,
    fecha_fin_str,
).main(filter=False)

if df_disponibilidad_planta is None or df_disponibilidad_planta.empty:
    raise ValueError(
        f"SIMEM no devolvió disponibilidad entre {fecha_inicio_str} y {fecha_fin_str}"
    )

required_columns = {
    "CodigoVariable",
    "FechaHora",
    "CodigoDuracion",
    "UnidadMedida",
    "CodigoPlanta",
    "Version",
    "Valor",
}
missing_columns = required_columns - set(df_disponibilidad_planta.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas esperadas: {sorted(missing_columns)}")

parsed_dates = pd.to_datetime(
    df_disponibilidad_planta["FechaHora"],
    errors="coerce",
)
if parsed_dates.isna().any():
    raise ValueError(
        f"Hay {int(parsed_dates.isna().sum()):,} valores inválidos en FechaHora"
    )

source_min_date = parsed_dates.min().date()
source_max_date = parsed_dates.max().date()
source_lag_days = (fecha_fin - source_max_date).days

natural_key = ["CodigoVariable", "FechaHora", "CodigoPlanta", "Version"]
duplicate_rows = int(df_disponibilidad_planta.duplicated(natural_key).sum())
if duplicate_rows > 0:
    raise ValueError(
        f"SIMEM devolvió {duplicate_rows:,} duplicados para la llave natural {natural_key}"
    )

if bronze_max_date is not None and source_max_date < bronze_max_date:
    raise ValueError(
        "La extracción termina antes que Bronze: "
        f"SIMEM={source_max_date}, Bronze={bronze_max_date}. No se publicará Landing."
    )

if source_lag_days > MAXIMUM_ACCEPTED_LAG_DAYS:
    raise ValueError(
        f"La fuente presenta {source_lag_days} días de rezago, por encima del umbral "
        f"de {MAXIMUM_ACCEPTED_LAG_DAYS}. No se publicará Landing."
    )

new_rows = (
    len(df_disponibilidad_planta)
    if bronze_max_date is None
    else int((parsed_dates.dt.date > bronze_max_date).sum())
)

validation_summary = pd.DataFrame([
    {
        "execution_mode": execution_mode,
        "requested_start_date": fecha_inicio,
        "requested_end_date": fecha_fin,
        "bronze_max_date_before": bronze_max_date,
        "source_min_date": source_min_date,
        "source_max_date": source_max_date,
        "source_lag_days": source_lag_days,
        "downloaded_rows": len(df_disponibilidad_planta),
        "new_rows_after_bronze_max": new_rows,
        "duplicate_rows": duplicate_rows,
    }
])
display(validation_summary)

In [0]:
# Publicación protegida: primero escribe un temporal y luego reemplaza el archivo objetivo.
timestamp = datetime.now(ZoneInfo("America/Bogota")).strftime("%Y%m%dT%H%M%S")
temporary_path = LANDING_PATH.replace(".json.gz", f".{timestamp}.tmp.json.gz")

df_disponibilidad_planta.to_json(
    temporary_path,
    orient="records",
    lines=True,
    mode="w",
    compression="gzip",
)

if not os.path.exists(temporary_path) or os.path.getsize(temporary_path) == 0:
    raise IOError(f"El archivo temporal no se creó correctamente: {temporary_path}")

os.replace(temporary_path, LANDING_PATH)
print(f"Landing publicado: {LANDING_PATH}")
print(f"Cobertura recibida: {source_min_date} a {source_max_date}")
print(f"Rezago observado de la fuente: {source_lag_days} días")